In [1]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:
root = Path("C:/UserData/Kaustav/python projects/GitaVani/dataset/raw")
ravdess = root/"ravdess"/"archive"
crema = root / "crema-d" / "archive" / "AudioWAV"
tess = root / "tess" / "archive" / "TESS Toronto emotional speech set data"

In [3]:
label_encoder = {
    "calm":0,
    "joy":1,
    "sadness":2,
    "anger":3,
    "fear":4
}

ravdess_map = {
    "02":"calm",
    "03":"joy",
    "04":"sadness",
    "05":"anger",
    "06":"fear"
}

crema_map = {
    "ANG":"anger",
    "HAP":"joy",
    "SAD":"sadness",
    "FEA":"fear"
}

In [5]:
print("Reading RAVDESS...")

for wav in tqdm(ravdess.rglob("*.wav")):

    # Example filename:
    # 03-01-05-01-01-01-01.wav

    parts = wav.stem.split("-")

    emotion_code = parts[2]
    speaker = parts[-1]

    if emotion_code not in ravdess_map:
        continue

    emotion = ravdess_map[emotion_code]

    metadata.append({
        "filepath": str(wav),
        "dataset": "RAVDESS",
        "speaker": speaker,
        "emotion": emotion,
        "label": label_encoder[emotion]
    })

print("Total samples after RAVDESS:", len(metadata))

Reading RAVDESS...


2880it [00:00, 21348.11it/s]

Total samples after RAVDESS: 1920


In [6]:
print("Reading CREMA-D...")

for wav in tqdm(crema.glob("*.wav")):

    # Example:
    # 1001_DFA_ANG_XX.wav

    parts = wav.stem.split("_")

    speaker = parts[0]
    emotion_code = parts[2]

    if emotion_code not in crema_map:
        continue

    emotion = crema_map[emotion_code]

    metadata.append({
        "filepath": str(wav),
        "dataset": "CREMA-D",
        "speaker": speaker,
        "emotion": emotion,
        "label": label_encoder[emotion]
    })

print("Total samples after CREMA-D:", len(metadata))

Reading CREMA-D...


7442it [00:00, 99920.64it/s]

Total samples after CREMA-D: 7004


In [9]:
print("Reading TESS...")

for folder in tqdm(tess.iterdir()):

    if not folder.is_dir():
        continue

    folder_name = folder.name.lower()

    emotion = None

    if "angry" in folder_name:
        emotion = "anger"

    elif "fear" in folder_name:
        emotion = "fear"

    elif "happy" in folder_name:
        emotion = "joy"

    elif "sad" in folder_name:
        emotion = "sadness"

    elif "pleasant_surprise" in folder_name or "pleasant_surprised" in folder_name:
        emotion = "joy"

    else:
        continue

    for wav in folder.glob("*.wav"):

        speaker = wav.stem.split("_")[0]

        metadata.append({
            "filepath": str(wav),
            "dataset": "TESS",
            "speaker": speaker,
            "emotion": emotion,
            "label": label_encoder[emotion]
        })

print("Total samples after TESS:", len(metadata))

Reading TESS...


15it [00:00, 1245.68it/s]

Total samples after TESS: 11004


In [12]:
df = pd.DataFrame(metadata)
df.head()

,filepath,dataset,speaker,emotion,label
0,C:\UserData\Kaustav\python projects\GitaVani\d...,RAVDESS,01,calm,0
1,C:\UserData\Kaustav\python projects\GitaVani\d...,RAVDESS,01,calm,0
2,C:\UserData\Kaustav\python projects\GitaVani\d...,RAVDESS,01,calm,0
3,C:\UserData\Kaustav\python projects\GitaVani\d...,RAVDESS,01,calm,0
4,C:\UserData\Kaustav\python projects\GitaVani\d...,RAVDESS,01,calm,0


In [18]:
df["emotion"].value_counts()
df["dataset"].value_counts()

dataset
CREMA-D    5084
TESS       4000
RAVDESS    1920
Name: count, dtype: int64

In [19]:
df.isnull().sum()

filepath    0
dataset     0
speaker     0
emotion     0
label       0
dtype: int64

In [20]:
df.groupby("dataset")["speaker"].nunique()

dataset
CREMA-D    91
RAVDESS    24
TESS        2
Name: speaker, dtype: int64

In [21]:
processed = root.parent / "processed"
processed.mkdir(exist_ok=True)
df.to_csv(processed / "metadata.csv", index=False)
print("Saved successfully!")

Saved successfully!


In [22]:
print(df["emotion"].value_counts())

print(df["dataset"].value_counts())

print(df.groupby("dataset")["speaker"].nunique())

emotion
joy        3255
sadness    2455
anger      2455
fear       2455
calm        384
Name: count, dtype: int64
dataset
CREMA-D    5084
TESS       4000
RAVDESS    1920
Name: count, dtype: int64
dataset
CREMA-D    91
RAVDESS    24
TESS        2
Name: speaker, dtype: int64
